# 25 · Kerala school access: nearest school and E2SFCA per level — Part D, step D5 (schools, part 2)

**What is computed for every populated 100 m cell in Kerala** (2.98 M cells; WorldPop 2026), per level
LP (Std 1–4), UP (5–7), HS (8–10), HSS (+1/+2):
1. **Nearest school** (m): exact great-circle distance to the nearest Government/Aided school of the level.
2. **Within the distance norm?** Is any school of the level within its catchment: LP 1 km, UP 3 km (RTE Model
   Rules 2010), HS 5 km, 3 km for hilly schools (Samagra Shiksha; Sametham hilly flag), HSS 7 km.
3. **E2SFCA score** = school places reachable per child, with distance decay (Luo & Qi 2009):
   - step 1, each school: **R_j = places_j / Σ_i children_i · w(d_ij)** over the cells in its catchment;
   - step 2, each cell: **A_i = Σ_j R_j · w(d_ij)** over the schools whose catchment it is in;
   - **w(d) = (e^(−½(d/d0)²) − e^(−½)) / (1 − e^(−½))** for d ≤ d0, else 0: Gaussian decay (Dai 2010), 1 at the
     school and 0 at the catchment edge d0, so **σ = d0**, no extra parameter.
   - Interpretation: A = 1 means one place per child; below 1, fewer places than children.

**Supply** (notebook 24): main run = norm-based places (KER / RTE rules × teachers; HSS batches × 50, VHSE × 30);
sensitivity runs = pupils enrolled, and HSS at 60 per batch. **Demand:** WorldPop children of the level's ages
× Government + Aided share (UDISE).

**How (Spark + Sedona):** the cell–school pairs come from a **broadcast index join** (`ST_DWithin` on the WGS84
ellipsoid; the fastest strategy in D4): the schools of a level, with an R-tree, go to every task; cells are not
moved. The pairs (up to ~30 M per level) are aggregated in the job and never written. Both supply versions use
the same pairs. Nearest school: a KD-tree of the schools in every task (`mapInPandas`); Sedona's `ST_KNN` join
was tried first and did not finish on Kerala in 10 minutes on this laptop.

**Checks:** Σ score × children = Σ places reached (the property E2SFCA guarantees); KD-tree = brute force on
2,000 cells; a cell with a school in reach has its nearest school within the catchment; main vs enrolment
results: rank correlation and overlap of the worst areas.

**Outputs** (`data/lake/gold/`): `school_access_kerala` (one row per cell), `school_ratio_kerala.csv` (per
school), `school_access_kerala_summary.csv`, `…_by_district.csv`, `…_by_local_body.csv`, `…_checks.csv`,
`…_main_vs_enrol.csv`.

**Limits:** straight-line distance; children cross the Kerala border neither way (Tamil Nadu / Karnataka schools
and children are not in this Kerala run; the India run covers them); 912 HS schools with an empty hilly flag use 5 km.

## Project folder (laptop or Colab)

In [1]:
# Project folder. Laptop: open the notebook from "PROJECTS IPYNB" (or the project root).
# Google Colab: upload the project folder to Google Drive ("My Drive") and set its name below.
from pathlib import Path
import sys
PROJECT_FOLDER = "Distance-Capacity-and-Inequality-in-Access-to-Government-and-Aided-Schools"  # Colab only: folder name inside My Drive
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
else:
    PROJECT = Path.cwd().parent if Path.cwd().name in ("notebooks", "PROJECTS IPYNB") else Path.cwd()
assert (PROJECT / "scripts").is_dir(), f"project folder not found: {PROJECT}"
sys.path.insert(0, str(PROJECT / "scripts"))

In [2]:
# Colab only: install packages that Colab does not have by default (does nothing on the laptop)
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz", "pymupdf", "pyogrio", "rasterio"], check=True)

## Install the Part D packages

In [3]:
# Part D packages, pinned in requirements-bigdata.txt (PySpark 3.5.9, apache-sedona 1.9.1, pyarrow).
# Colab also needs Java 17, which Spark runs on (the laptop already has it: JAVA_HOME).
import os
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements-bigdata.txt")], check=True)
if IN_COLAB:
    subprocess.run("apt-get install -y -qq openjdk-17-jdk-headless > /dev/null", shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

## Start Spark + Sedona

In [4]:
import shutil
import time

import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from scipy.spatial import cKDTree
from bigdata.spark_session import get_spark

sys.path.insert(0, str(PROJECT / "scripts" / "bigdata"))
import population_cells as pc
import school_capacity as sc

SILVER, GOLD = PROJECT / "data" / "lake" / "silver", PROJECT / "data" / "lake" / "gold"
LEVELS = ["lp", "up", "hs", "hss"]
VARIANTS = {"main": "cap_main", "enrol": "cap_enrol", "hss60": "cap_hss60"}  # hss60 differs from main only for HSS
EARTH_R = 6371000.0  # Earth radius (m) of the project's haversine_m()
assert shutil.disk_usage(PROJECT).free / 1e9 > 5, "less than 5 GB free: stop"
spark = get_spark(PROJECT, cores=6, memory="6g")  # 6 cores: the laptop's free RAM (see notebook 17)
LON, LAT = pc.lonlat_sql(PROJECT / "data" / "raw" / "worldpop")
print("cores:", spark.sparkContext.defaultParallelism, "| Spark UI:", spark.sparkContext.uiWebUrl)

cores: 6 | Spark UI: http://127.0.0.1:4040


## Demand (cells) and supply (schools)

In [5]:
# Demand: children of each level's ages in every populated 100 m cell x Government + Aided share (notebook 24).
# Cells: Kerala + the Mahe enclave (Puducherry, west of 76 E), whose schools are in the Kerala school list.
share = pd.read_parquet(SILVER / "school_demand_share").query("scope == 'kerala'").set_index("level")["share"]
cells = (spark.read.parquet(str(SILVER / "population_cells")).filter("state IN ('Kerala', 'Puducherry')")
         .withColumn("lon", F.expr(LON).cast("double")).withColumn("lat", F.expr(LAT).cast("double")).filter("state = 'Kerala' OR lon < 76")
         .selectExpr("cell_id", "state", "district_code", "local_body_code", "pop", "lon", "lat", "ST_Point(lon, lat) AS geom",
                     *[f"coalesce({sc.demand_sql(lv)}, 0) * {float(share[lv])!r} AS dem_{lv}" for lv in LEVELS])
         .cache())
# Supply: places per school and level (notebook 24)
sup_pd = pd.read_parquet(SILVER / "school_supply_kerala")
sup_pd["cap_hss60"] = sup_pd["cap_hss60"].fillna(sup_pd["cap_main"])
sup = spark.createDataFrame(sup_pd[["school_id", "level", "catchment_m", "lat", "lon", *VARIANTS.values()]]) \
           .selectExpr("*", "ST_Point(lon, lat) AS geom").cache()
tot = cells.agg(F.count("*").alias("cells"), F.sum("pop").alias("pop"), *[F.sum(f"dem_{lv}").alias(lv) for lv in LEVELS]).first()
print(f"{tot['cells']:,} cells, {tot['pop'] / 1e6:.2f} M people; Government + Aided children per level (M):",
      {lv: round(tot[lv] / 1e6, 3) for lv in LEVELS})
print("school-level rows:", sup_pd.groupby("level").size().to_dict(), "| shares:", share.round(3).to_dict())

2,979,016 cells, 37.53 M people; Government + Aided children per level (M): {'lp': 0.995, 'up': 0.961, 'hs': 1.106, 'hss': 0.835}
school-level rows: {'hs': 2714, 'hss': 1899, 'lp': 9392, 'up': 5696} | shares: {'lp': 0.611, 'up': 0.71, 'hs': 0.801, 'hss': 0.875}


## Pairs, E2SFCA and nearest school

In [6]:
def pairs(level):
    """Every (cell, school) pair within the school's catchment, with distance (m, WGS84 ellipsoid) and Gaussian weight.
    Broadcast index join (fastest in D4): the level's schools go to every task with an R-tree; cells stay in place.
    One join per catchment distance (HS: 3 km hilly, 5 km plains)."""
    s = sup.filter(F.col("level") == level)
    cells.createOrReplaceTempView("c")
    out = []
    for d0 in sorted(r[0] for r in s.select("catchment_m").distinct().collect()):
        s.filter(F.col("catchment_m") == d0).createOrReplaceTempView(f"s{d0}")
        out.append(spark.sql(f"""SELECT /*+ BROADCAST(s) */ c.cell_id, c.dem_{level} AS dem, s.school_id, s.catchment_m AS d0,
                                        {", ".join("s." + v for v in VARIANTS.values())}, ST_DistanceSpheroid(c.geom, s.geom) AS d
                                 FROM c JOIN s{d0} s ON ST_DWithin(c.geom, s.geom, {d0}, true)"""))
    p = out[0] if len(out) == 1 else out[0].unionByName(out[1])
    return p.withColumn("w", F.expr(sc.gaussian("d", "d0")))


def e2sfca(level):
    """Step 1: ratio R_j = places_j / sum_i(children_i x w_ij) per school.
    Step 2: score A_i = sum_j(R_j x w_ij) per cell = places per child reachable from the cell."""
    p = pairs(level)
    plan = p._jdf.queryExecution().executedPlan().toString()
    assert "BroadcastIndexJoin" in plan, f"{level}: Sedona did not use the broadcast index join"
    sch = p.groupBy("school_id").agg(F.sum(F.col("dem") * F.col("w")).alias("weighted_children"), F.count("*").alias("cells_in_catchment"),
                                     *[F.first(c).alias(c) for c in VARIANTS.values()])
    for v, c in VARIANTS.items():
        sch = sch.withColumn(f"r_{v}", F.col(c) / F.col("weighted_children"))  # null when no child lives in the catchment
    sch = sch.cache()  # ratios stay in Spark and are broadcast to the cells (no copy through the driver)
    r = sch.select("school_id", *[f"r_{v}" for v in VARIANTS])
    cell = (p.join(F.broadcast(r), "school_id").groupBy("cell_id")
             .agg(*[F.sum(F.col(f"r_{v}") * F.col("w")).alias(f"a_{v}") for v in VARIANTS], F.count("*").alias("schools_in_reach"),
                  F.min("d").alias("nearest_in_reach_m")))
    return sch, cell


def nearest(level):
    """Distance (m, sphere) from every cell to the nearest school of the level: a KD-tree of the schools' unit vectors
    is shipped to the Spark tasks (mapInPandas); chord length -> great-circle distance. Exact, no search radius."""
    s = sup_pd[sup_pd["level"] == level]
    xyz = lambda lat, lon: np.column_stack([np.cos(np.radians(lat)) * np.cos(np.radians(lon)),
                                            np.cos(np.radians(lat)) * np.sin(np.radians(lon)), np.sin(np.radians(lat))])
    tree = cKDTree(xyz(s["lat"].to_numpy(), s["lon"].to_numpy()))

    def f(batches):
        for b in batches:
            chord, _ = tree.query(xyz(b["lat"].to_numpy(), b["lon"].to_numpy()))
            yield pd.DataFrame({"cell_id": b["cell_id"], "nearest_m": 2 * EARTH_R * np.arcsin(np.minimum(chord / 2, 1))})
    return cells.select("cell_id", "lat", "lon").mapInPandas(f, "cell_id long, nearest_m double"), tree, xyz

## Run all four levels (+ check: places are conserved)

In [7]:
from build_india_health_facilities import haversine_m

out, ratios, checks = cells.select("cell_id", "state", "district_code", "local_body_code", "pop", "lon", "lat",
                                   *[f"dem_{lv}" for lv in LEVELS]), [], []
for level in LEVELS:
    t = time.time()
    sch, cell = e2sfca(level)
    near, tree, xyz = nearest(level)
    keep = ["a_main", "a_enrol", "a_hss60"] if level == "hss" else ["a_main", "a_enrol"]
    cell = cell.select("cell_id", *[F.col(a).alias(f"{a}_{level}") for a in keep],
                       F.col("schools_in_reach").alias(f"schools_in_reach_{level}"), F.col("nearest_in_reach_m").alias(f"nearest_in_reach_{level}"))
    new = out.join(cell, "cell_id", "left").join(near.withColumnRenamed("nearest_m", f"nearest_m_{level}"), "cell_id", "left").cache()
    new.count()  # materialise this level once, so later levels do not recompute its joins
    out.unpersist()
    out = new
    sch_spark, sch = sch, sch.toPandas().assign(level=level)
    sch_spark.unpersist()
    ratios.append(sch)
    # check 1 (the property E2SFCA guarantees): sum over cells of score x children = places of schools with children in reach
    got = out.agg(*[F.sum(F.col(f"{a}_{level}") * F.col(f"dem_{level}")).alias(a) for a in keep]).first()
    for a in keep:
        v = a[2:]
        want = sch.loc[sch[f"r_{v}"].notna(), VARIANTS[v]].sum()
        checks.append({"level": level, "variant": v, "sum_score_x_children": got[a], "places_reached": want,
                       "rel_diff": abs(got[a] - want) / want})
    print(f"{level}: {len(sch):,} schools with cells in reach, {time.time() - t:.0f} s")
checks = pd.DataFrame(checks)
print(checks.to_string(index=False))
assert (checks["rel_diff"] < 1e-6).all(), "E2SFCA does not conserve places"

lp: 9,390 schools with cells in reach, 43 s


up: 5,696 schools with cells in reach, 156 s


hs: 2,714 schools with cells in reach, 206 s


hss: 1,899 schools with cells in reach, 263 s
level variant  sum_score_x_children  places_reached     rel_diff
   lp    main          1.316404e+06    1.316404e+06 3.537373e-16
   lp   enrol          9.604610e+05    9.604610e+05 4.848310e-16
   up    main          9.517399e+05    9.517399e+05 4.892737e-16
   up   enrol          9.283490e+05    9.283490e+05 1.254004e-16
   hs    main          1.506775e+06    1.506775e+06 1.545225e-16
   hs   enrol          1.141074e+06    1.141074e+06 4.080903e-16
  hss    main          6.851800e+05    6.851800e+05 1.699047e-16
  hss   enrol          7.926580e+05    7.926580e+05 7.343351e-16
  hss   hss60          8.092200e+05    8.092200e+05 5.754446e-16


## Check the nearest-school distances

In [8]:
# check 2: KD-tree nearest = brute force (haversine to every school) on 2,000 random cells per level
# check 3: a cell with a school in its catchment has its nearest school within that catchment. The catchment test uses the
# WGS84 ellipsoid, the nearest distance a sphere (R 6371 km); north-south at Kerala's latitude the ellipsoid's radius
# is ~6337 km, so a sphere distance can be up to 0.54 % longer: allowed 0.6 %.
sample = out.sample(fraction=2500 / tot["cells"], seed=1).limit(2000).toPandas()
for level in LEVELS:
    s = sup_pd[sup_pd["level"] == level]
    brute = np.array([haversine_m(la, lo, s["lat"].to_numpy(), s["lon"].to_numpy()).min() for la, lo in zip(sample["lat"], sample["lon"])])
    err = np.abs(sample[f"nearest_m_{level}"] - brute).max()
    d0 = s["catchment_m"].max()
    bad = out.filter(f"schools_in_reach_{level} > 0 AND nearest_m_{level} > 1.006 * {d0}").count()
    print(f"{level}: KD-tree vs brute force, largest difference {err:.4f} m | cells in reach but nearest beyond catchment: {bad}")
    assert err < 0.01 and bad == 0

lp: KD-tree vs brute force, largest difference 0.0000 m | cells in reach but nearest beyond catchment: 0


up: KD-tree vs brute force, largest difference 0.0000 m | cells in reach but nearest beyond catchment: 0


hs: KD-tree vs brute force, largest difference 0.0000 m | cells in reach but nearest beyond catchment: 0


hss: KD-tree vs brute force, largest difference 0.0000 m | cells in reach but nearest beyond catchment: 0


## Results for Kerala

In [9]:
# Kerala results per level (children-weighted: each cell counts by its children of that level's ages)
norm_km = {"lp": "1", "up": "3", "hs": "5 (3 hilly)", "hss": "7"}
rows = []
for level in LEVELS:
    d = f"dem_{level}"
    r = out.agg(F.sum(d).alias("children"),
                F.sum(F.when(F.col(f"schools_in_reach_{level}").isNull(), F.col(d)).otherwise(0)).alias("beyond"),
                (F.sum(F.col(d) * F.col(f"nearest_m_{level}")) / F.sum(d)).alias("mean_nearest_m"),
                (F.sum(F.col(d) * F.coalesce(F.col(f"a_main_{level}"), F.lit(0))) / F.sum(d)).alias("a_main"),
                (F.sum(F.col(d) * F.coalesce(F.col(f"a_enrol_{level}"), F.lit(0))) / F.sum(d)).alias("a_enrol"),
                F.expr(f"percentile_approx(nearest_m_{level}, 0.99)").alias("p99_nearest_m")).first()
    rows.append({"level": level, "norm_km": norm_km[level], "GA_children": round(r["children"]),
                 "children_beyond_norm_%": round(100 * r["beyond"] / r["children"], 2), "mean_nearest_km": round(r["mean_nearest_m"] / 1000, 2),
                 "p99_cell_nearest_km": round(r["p99_nearest_m"] / 1000, 2),
                 "places_per_child_main": round(r["a_main"], 3), "places_per_child_enrol": round(r["a_enrol"], 3)})
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

level     norm_km  GA_children  children_beyond_norm_%  mean_nearest_km  p99_cell_nearest_km  places_per_child_main  places_per_child_enrol
   lp           1       995091                   19.29             0.69                 3.01                  1.323                   0.965
   up           3       961221                    1.00             0.91                 4.37                  0.990                   0.966
   hs 5 (3 hilly)      1106069                    0.49             1.29                 5.07                  1.362                   1.032
  hss           7       835281                    0.13             1.52                 6.28                  0.820                   0.949


## Districts, local bodies, and main vs enrolment run

In [10]:
# Per district and per local body (children-weighted means), and how well the two capacity versions agree
from scipy.stats import spearmanr

names = {u: pc.load_units(SILVER, f"{u}_code").set_index("code")["unit_name"] for u in ["district", "local_body"]}
area_rows = {}
for unit in ["district", "local_body"]:
    aggs = []
    for level in LEVELS:
        d = F.col(f"dem_{level}")
        aggs += [F.sum(d).alias(f"children_{level}"),
                 (100 * F.sum(F.when(F.col(f"schools_in_reach_{level}").isNull(), d).otherwise(0)) / F.sum(d)).alias(f"beyond_norm_%_{level}"),
                 (F.sum(d * F.col(f"nearest_m_{level}")) / F.sum(d) / 1000).alias(f"mean_nearest_km_{level}"),
                 (F.sum(d * F.coalesce(F.col(f"a_main_{level}"), F.lit(0))) / F.sum(d)).alias(f"a_main_{level}"),
                 (F.sum(d * F.coalesce(F.col(f"a_enrol_{level}"), F.lit(0))) / F.sum(d)).alias(f"a_enrol_{level}")]
    a = out.groupBy(f"{unit}_code").agg(F.sum("pop").alias("pop"), *aggs).toPandas()
    a.insert(1, unit, a[f"{unit}_code"].map(names[unit]).fillna("(none: Mahe or no OSM local body)"))
    area_rows[unit] = a.sort_values(f"{unit}_code").reset_index(drop=True)

agree = []
for unit, a in area_rows.items():
    for level in LEVELS:
        x = a[a[f"children_{level}"] >= 100]  # areas with at least 100 children of the level
        n_worst = 3 if unit == "district" else 10
        w_main = set(x.nsmallest(n_worst, f"a_main_{level}")[unit])
        w_enrol = set(x.nsmallest(n_worst, f"a_enrol_{level}")[unit])
        agree.append({"unit": unit, "level": level, "areas": len(x),
                      "spearman_main_vs_enrol": round(spearmanr(x[f"a_main_{level}"], x[f"a_enrol_{level}"]).statistic, 3),
                      f"worst_overlap": f"{len(w_main & w_enrol)} of {n_worst}"})
agree = pd.DataFrame(agree)
print(agree.to_string(index=False))
cols = ["district"] + [f"{m}_{lv}" for lv in LEVELS for m in ("beyond_norm_%", "a_main")]
print("\nper district:")
print(area_rows["district"][cols].round(3).to_string(index=False))

      unit level  areas  spearman_main_vs_enrol worst_overlap
  district    lp     17                   0.909        2 of 3
  district    up     17                   0.971        2 of 3
  district    hs     17                   0.934        2 of 3
  district   hss     17                   0.775        2 of 3
local_body    lp   1025                   0.897       4 of 10
local_body    up   1025                   0.936       6 of 10
local_body    hs   1027                   0.950       5 of 10
local_body   hss   1023                   0.889       7 of 10

per district:
          district  beyond_norm_%_lp  a_main_lp  beyond_norm_%_up  a_main_up  beyond_norm_%_hs  a_main_hs  beyond_norm_%_hss  a_main_hss
          Palakkad            26.805      1.396             4.767      1.074             1.692      1.503              0.339       0.765
      The Nilgiris            83.270      0.331            30.118      0.559            51.280      0.613              0.313       0.569
          Kottay

## Save

In [11]:
GOLD.mkdir(parents=True, exist_ok=True)
t = time.time()
out.write.mode("overwrite").option("compression", "zstd").parquet(str(GOLD / "school_access_kerala"))
pd.concat(ratios, ignore_index=True).to_csv(GOLD / "school_ratio_kerala.csv", index=False)
summary.to_csv(GOLD / "school_access_kerala_summary.csv", index=False)
checks.to_csv(GOLD / "school_access_kerala_checks.csv", index=False)
agree.to_csv(GOLD / "school_access_kerala_main_vs_enrol.csv", index=False)
for unit, a in area_rows.items():
    a.to_csv(GOLD / f"school_access_kerala_by_{unit}.csv", index=False)
n = spark.read.parquet(str(GOLD / "school_access_kerala")).count()
assert n == tot["cells"], (n, tot["cells"])
print(f"written gold/school_access_kerala ({n:,} cells) + 6 tables in {time.time() - t:.0f} s")
spark.stop()

written gold/school_access_kerala (2,979,016 cells) + 6 tables in 27 s
